# 02 · Prediction or lookup?

Before an agent can hand a prediction off, it has to notice it's being asked for one.

Not every question about a table is a prediction:

- *"How many customers churned last year?"* — a **lookup**. The answer is in the data; a `groupby` gets it.
- *"Which of our current customers will churn?"* — a **prediction**. The answer is *not* in the data; something has to be fit.

Get this wrong in one direction and the agent fits a model to answer a `COUNT(*)`. Get it wrong in the other and it answers a prediction question by, well, guessing — which is exactly what notebook 01 showed goes badly.

In this notebook we make the decision with an LLM. In notebook 04 we'll swap in a model that's purpose-built for decisions like this (Jev) and compare.

In [ ]:
# Run from the repo root or from notebooks/ — this makes `agent` importable either way
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "agent").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

# Keys: found in your environment or .env → used as-is; otherwise you're prompted (input hidden, nothing saved to disk)
from agent.keys import ensure_keys
ensure_keys("TABPFN_TOKEN", "ANTHROPIC_API_KEY")

TABPFN_TOKEN       found in environment
ANTHROPIC_API_KEY  found in environment


{'TABPFN_TOKEN': 'env', 'ANTHROPIC_API_KEY': 'env'}

## 1. The classifier: a JSON schema with an enum

The cleanest way to get a *label* out of an LLM is to constrain its output: we give the API a JSON schema where `task_type` is an `enum`, and the reply is guaranteed to match it — no parsing, no "Sure! The answer is…".

In [ ]:
import anthropic, os, time, json
client = anthropic.Anthropic()
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5-5")

TASK_TYPES = {
    "prediction": "The user wants an estimate of something NOT in the data: a future outcome, a probability, a score for specific rows. Words like predict, likely, will, risk, forecast, expected. Answering requires fitting a model.",
    "lookup":     "The user wants a fact computable directly from the data as it is: counts, averages, shares, filters, group-bys. Words like how many, average, list, show me. Answering only requires a query.",
    "other":      "Neither: chit-chat, a question about the tools, or unrelated to the dataset.",
}

CLASSIFY_SCHEMA = {
    "type": "object",
    "properties": {
        "task_type": {"type": "string", "enum": list(TASK_TYPES),
                      "description": "\n".join(f"{k}: {v}" for k, v in TASK_TYPES.items())},
        "reason": {"type": "string", "description": "One short sentence."},
    },
    "required": ["task_type", "reason"],
    "additionalProperties": False,
}

SYSTEM = ("Classify what kind of task the user's question is, for an assistant that has a table "
          "of telecom customers and whether each churned. Task types:\n"
          + "\n".join(f"- {k}: {v}" for k, v in TASK_TYPES.items()))

def classify_with_llm(question: str) -> dict:
    msg = client.messages.create(
        model=MODEL, max_tokens=300, system=SYSTEM,
        messages=[{"role": "user", "content": question}],
        output_config={"format": {"type": "json_schema", "schema": CLASSIFY_SCHEMA}},   # <- the reply MUST match this schema
    )
    text = next(b.text for b in msg.content if b.type == "text")   # skip thinking blocks
    return json.loads(text)

classify_with_llm("Which of our current customers are most likely to churn?")

{'task_type': 'prediction',
 'reason': 'The user wants churn risk scores for current customers, which requires fitting a model to estimate a future outcome not present in the data.'}

## 2. Try it on a batch

Some of these are easy. Some are deliberately annoying: lookups that *sound* predictive, predictions phrased as lookups.

In [ ]:
QUESTIONS = [
    ("How many customers do we have, and what share of them churned?",          "lookup"),
    ("Which of our current customers are most likely to churn? Top 10.",         "prediction"),
    ("What's the average monthly charge for customers on a two-year contract?",   "lookup"),
    ("Is this customer likely to churn? Female, 0, Yes, Yes, 1, Yes, Yes, 'Fiber optic', ...", "prediction"),
    ("List everyone who churned and was paying by electronic check.",            "lookup"),
    ("If we moved all month-to-month customers to annual contracts, who would still leave?", "prediction"),
    ("What was our churn rate among fiber optic customers?",                     "lookup"),
    ("Score our customer base for churn risk.",                                  "prediction"),
    ("Show me customers with tenure under 3 months.",                            "lookup"),
    ("How does TabPFN work?",                                                    "other"),
    ("Which customers will probably churn — I mean the ones who already did, how many?", "lookup"),
    ("Thanks, that's all!",                                                      "other"),
]

import pandas as pd
rows, t0 = [], time.perf_counter()
for q, expected in QUESTIONS:
    out = classify_with_llm(q)
    rows.append({"question": q[:70], "expected": expected, "llm": out["task_type"], "ok": out["task_type"] == expected, "reason": out["reason"]})
llm_batch_seconds = time.perf_counter() - t0

results = pd.DataFrame(rows)
print(f"{results.ok.sum()}/{len(results)} correct in {llm_batch_seconds:.1f}s ({llm_batch_seconds/len(results):.2f}s per question)")
pd.set_option("display.max_colwidth", 80)
results

12/12 correct in 21.6s (1.80s per question)


,question,expected,llm,ok,reason
0,"How many customers do we have, and what share of them churned?",lookup,lookup,True,Counting customers and computing the churn share can be done directly from t...
1,Which of our current customers are most likely to churn? Top 10.,prediction,prediction,True,Ranking current customers by their likelihood to churn requires a model to e...
2,What's the average monthly charge for customers on a two-year contract,lookup,lookup,True,"Asks for an average of monthly charges filtered by contract type, which is a..."
3,"Is this customer likely to churn? Female, 0, Yes, Yes, 1, Yes, Yes, 'F",prediction,prediction,True,"The user wants the churn likelihood for a specific customer's attributes, wh..."
4,List everyone who churned and was paying by electronic check.,lookup,lookup,True,The user wants a filtered list of existing rows (churned customers paying by...
5,"If we moved all month-to-month customers to annual contracts, who woul",prediction,prediction,True,It asks for a hypothetical scenario outcome (who would still churn after a c...
6,What was our churn rate among fiber optic customers?,lookup,lookup,True,Churn rate among fiber optic customers is a historical share computable dire...
7,Score our customer base for churn risk.,prediction,prediction,True,"The user wants a churn risk score for each customer, which is an estimate no..."
8,Show me customers with tenure under 3 months.,lookup,lookup,True,Filtering customers by tenure under 3 months is a direct query on existing d...
9,How does TabPFN work?,other,other,True,"This asks how a tool (TabPFN) works, which is neither a data query nor a pre..."


## 3. What's missing

The LLM is probably getting most of these right. But look at what you get back: a **label**. Not *how sure* it is.

For a router that's a real problem. When the question is "which customers will *probably* churn — I mean the ones who already did", we'd like the system to know it's on a knife-edge and ask, not confidently pick one. You can ask the LLM for a confidence number, but — as notebook 01 showed for churn probabilities — a number an LLM writes down isn't a calibrated probability, it's a number it wrote down.

The other thing: every one of these decisions is a full LLM round-trip. Fine for a demo; not fine as a gate in front of every message in production.

This is the same shape as the LLM-vs-TFM lesson, one level up: **classification with a confidence is a job for a model built for it.** That's what [Jev](https://typesafe.ai) is — a *decision model*: you give it text and a typed question, it returns a probability distribution over the options, in milliseconds. We'll use it as the router in notebook 04.

## 4. Save the classifier

The same classifier lives in `agent/router.py` as the Claude fallback, so the agent in notebook 04 works even without a Jev key:

In [ ]:
from agent.router import route
r = route("How many fiber optic customers churned?", router="claude")
r

Route(task_type='lookup', probabilities={'lookup': 1.0}, router='claude')

Next: [03 · the handoff](../README.md#3-hand-prediction-tasks-to-tabpfn-from-a-coding-agent) is done in Claude Code rather than a notebook — open the README for the walkthrough. Then [04 · the predictive agent](04_predictive_agent.ipynb) puts it all together.